In [1]:
# Dataset: https://www.kaggle.com/datasets/rmisra/news-category-dataset

In [1]:
import pandas as pd
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense 
from sklearn.model_selection import train_test_split
import pickle

### Date gathering

In [3]:
data = pd.read_json('News_Category_Dataset_v3.json', lines=True)

### Data exploration

In [4]:
data

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22
...,...,...,...,...,...,...
209522,https://www.huffingtonpost.com/entry/rim-ceo-t...,RIM CEO Thorsten Heins' 'Significant' Plans Fo...,TECH,Verizon Wireless and AT&T are already promotin...,"Reuters, Reuters",2012-01-28
209523,https://www.huffingtonpost.com/entry/maria-sha...,Maria Sharapova Stunned By Victoria Azarenka I...,SPORTS,"Afterward, Azarenka, more effusive with the pr...",,2012-01-28
209524,https://www.huffingtonpost.com/entry/super-bow...,"Giants Over Patriots, Jets Over Colts Among M...",SPORTS,"Leading up to Super Bowl XLVI, the most talked...",,2012-01-28
209525,https://www.huffingtonpost.com/entry/aldon-smi...,Aldon Smith Arrested: 49ers Linebacker Busted ...,SPORTS,CORRECTION: An earlier version of this story i...,,2012-01-28


In [5]:
data.shape

(209527, 6)

In [6]:
data.columns

Index(['link', 'headline', 'category', 'short_description', 'authors', 'date'], dtype='str')

In [7]:
# We need only the short description for our next word prediction task. So we will drop the other columns.
data = data[['short_description']]

In [8]:
data

,short_description
0,Health experts said it is too early to predict...
1,He was subdued by passengers and crew when he ...
2,"""Until you have a dog you don't understand wha..."
3,"""Accidentally put grown-up toothpaste on my to..."
4,Amy Cooper accused investment firm Franklin Te...
...,...
209522,Verizon Wireless and AT&T are already promotin...
209523,"Afterward, Azarenka, more effusive with the pr..."
209524,"Leading up to Super Bowl XLVI, the most talked..."
209525,CORRECTION: An earlier version of this story i...


### Check null and duplicate columns and remove it

In [9]:
# Check null values
data.isnull().sum()

short_description    0
dtype: int64

In [10]:
# Check duplicates
data.duplicated().sum()

np.int64(22505)

In [11]:
# Drop duplicates
data.drop_duplicates(inplace=True)

In [12]:
# Check duplicates after remove it
data.duplicated().sum()

np.int64(0)

### Date preprocessing

In [13]:
tokenizer = Tokenizer(oov_token='<OOV>')

In [14]:
# Fit the tokenizer on the short descriptions
tokenizer.fit_on_texts(data['short_description'])

In [15]:
tokenizer.word_index

{'<OOV>': 1,
 'the': 2,
 'to': 3,
 'a': 4,
 'of': 5,
 'and': 6,
 'in': 7,
 'is': 8,
 'that': 9,
 'for': 10,
 'i': 11,
 'you': 12,
 'on': 13,
 'it': 14,
 'with': 15,
 'are': 16,
 'we': 17,
 'be': 18,
 'this': 19,
 'as': 20,
 'have': 21,
 'was': 22,
 'but': 23,
 'not': 24,
 'your': 25,
 'at': 26,
 'from': 27,
 'my': 28,
 'an': 29,
 'or': 30,
 'about': 31,
 'has': 32,
 'more': 33,
 'our': 34,
 'by': 35,
 'can': 36,
 'one': 37,
 'what': 38,
 'all': 39,
 'when': 40,
 'their': 41,
 'will': 42,
 'they': 43,
 'out': 44,
 'who': 45,
 "it's": 46,
 'his': 47,
 'if': 48,
 'new': 49,
 'he': 50,
 'time': 51,
 'just': 52,
 'up': 53,
 'people': 54,
 'so': 55,
 'her': 56,
 'do': 57,
 'like': 58,
 'how': 59,
 'some': 60,
 'there': 61,
 'been': 62,
 'said': 63,
 'no': 64,
 'than': 65,
 'us': 66,
 'me': 67,
 'most': 68,
 'year': 69,
 'after': 70,
 'get': 71,
 'day': 72,
 'life': 73,
 'she': 74,
 'had': 75,
 'many': 76,
 'would': 77,
 'into': 78,
 'were': 79,
 'make': 80,
 'these': 81,
 'first': 82,
 'over

In [31]:
# Count of unique words in the dataset
vocab_size=len(tokenizer.word_index)

In [17]:
# Convert the short descriptions to sequences of integers
tokenizer.texts_to_sequences(data['short_description'])

[[134,
  926,
  63,
  14,
  8,
  137,
  444,
  3,
  5238,
  212,
  2018,
  77,
  2036,
  53,
  15,
  2,
  27536,
  310,
  8784,
  5,
  2,
  49,
  27537,
  2,
  133,
  131,
  2888,
  10,
  2,
  482],
 [50,
  22,
  19731,
  35,
  2951,
  6,
  3098,
  40,
  50,
  5659,
  3,
  2,
  115,
  5,
  2,
  5743,
  70,
  2,
  8311,
  295,
  3,
  2,
  133,
  131,
  16637,
  519,
  7,
  1095,
  1172],
 [374, 12, 21, 4, 947, 12, 93, 523, 38, 98, 18, 4988],
 [5591,
  245,
  1834,
  53,
  15871,
  13,
  28,
  37175,
  24911,
  6,
  50,
  13559,
  58,
  11,
  22,
  2878,
  47,
  3440,
  15,
  4,
  1777,
  31409,
  14059,
  7,
  31410,
  3058],
 [3032,
  4989,
  869,
  2373,
  2776,
  6694,
  48042,
  5,
  11814,
  5131,
  56,
  6,
  7577,
  56,
  4,
  2259,
  70,
  280,
  5,
  2,
  1313,
  911,
  3045,
  500,
  2362],
 [2,
  10818,
  69,
  125,
  307,
  22,
  393,
  356,
  26,
  2,
  665,
  1777,
  1072,
  13,
  905,
  74,
  22,
  208,
  826,
  752,
  70,
  56,
  156,
  845,
  56,
  1285,
  1303,
  63],


In [18]:
input_sequences = []
for line in data['short_description']:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

In [19]:
input_sequences

[[134, 926],
 [134, 926, 63],
 [134, 926, 63, 14],
 [134, 926, 63, 14, 8],
 [134, 926, 63, 14, 8, 137],
 [134, 926, 63, 14, 8, 137, 444],
 [134, 926, 63, 14, 8, 137, 444, 3],
 [134, 926, 63, 14, 8, 137, 444, 3, 5238],
 [134, 926, 63, 14, 8, 137, 444, 3, 5238, 212],
 [134, 926, 63, 14, 8, 137, 444, 3, 5238, 212, 2018],
 [134, 926, 63, 14, 8, 137, 444, 3, 5238, 212, 2018, 77],
 [134, 926, 63, 14, 8, 137, 444, 3, 5238, 212, 2018, 77, 2036],
 [134, 926, 63, 14, 8, 137, 444, 3, 5238, 212, 2018, 77, 2036, 53],
 [134, 926, 63, 14, 8, 137, 444, 3, 5238, 212, 2018, 77, 2036, 53, 15],
 [134, 926, 63, 14, 8, 137, 444, 3, 5238, 212, 2018, 77, 2036, 53, 15, 2],
 [134,
  926,
  63,
  14,
  8,
  137,
  444,
  3,
  5238,
  212,
  2018,
  77,
  2036,
  53,
  15,
  2,
  27536],
 [134,
  926,
  63,
  14,
  8,
  137,
  444,
  3,
  5238,
  212,
  2018,
  77,
  2036,
  53,
  15,
  2,
  27536,
  310],
 [134,
  926,
  63,
  14,
  8,
  137,
  444,
  3,
  5238,
  212,
  2018,
  77,
  2036,
  53,
  15,
  2,
  27

In [20]:
max_len = max(len(x) for x in input_sequences)

In [21]:
max_len

246

In [22]:
input = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

In [27]:
# All row 
X = input[:,:-1]
# Final word of each row
y = np.array(input[:,-1])

### Train Test split

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [29]:
X_train.shape, y_train.shape

((3150234, 245), (3150234,))

In [30]:
max_len = X.shape[1]

### Create a model

In [32]:
model = Sequential([
    Embedding(vocab_size, 128, input_length=max_len),
    LSTM(256),
    Dense(vocab_size, activation='softmax')
])

e:\GenAI\projects\next-word-prediction-using-lstm\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [33]:
model.compile(
    loss='sparse_categorical_crossentropy', 
    optimizer='adam', 
    metrics=['accuracy']
    )

In [34]:
model.build(input_shape=(None, max_len))

In [35]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 245, 128)       │    11,455,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 89495)          │    23,000,215 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,849,815 (132.94 MB)

 Trainable params: 34,849,815 (132.94 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))

Epoch 1/10
  266/98445 ━━━━━━━━━━━━━━━━━━━━ 46:09:00 2s/step - accuracy: 0.0408 - loss: 9.5032

In [ ]:
model.save('nextword.h5')

In [ ]:
with open('tokenzier.pkl', 'wb') as file:
    pickle.dump(tokenizer, file)